# Find the best overlapping WAC–NAC image pairs

Scan **every `*.tif` file** in the WAC collection and both NAC collections,
using **10 worker processes for both footprint reading and pair ranking**. Search the **full WAC footprints**, with no
restriction to the earlier poster AOI. WAC VIS and UV files are separate entries.

**Overlap (%) = 100 × area(WAC footprint ∩ NAC footprint) / area(WAC footprint).**

The result is an **M × N nested dictionary**: every WAC filename maps to a
dictionary containing every NAC filename across both directories. Values are
precise percentages, `0.0` for no area overlap, or `None` if either raster
could not be inspected. The last cell prints the **top five pairs overall**,
across all WAC VIS/UV and NAC files.

Only raster metadata is read. Boundaries follow the affine outer pixel edges,
including rotation, and are densified before transformation to the repository's
IAU:30100 lunar CRS. Surface areas use the lunar sphere, not square degrees.
**NoData margins count toward the footprint**; inspect imagery before labeling.
Shared rasters and indexes remain read-only.

## 1. Imports and repository discovery

Run from the repository root or its top-level `notebooks/` directory.
The worker is an importable Python function so multiprocessing with `spawn`
works in Jupyter. Dependencies are already in `requirements_container.txt`:
NumPy, Rasterio, PyProj, Shapely, and tqdm. No training libraries are needed.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime, timezone
from functools import partial
import json
import multiprocessing
from pathlib import Path
import sys
from time import perf_counter
from tqdm import tqdm

working_dir = Path(str(Path.cwd()).replace(
    "/panfs/ccds02/nobackup", "/explore/nobackup"
))
repo_root = next(
    (p for p in (working_dir, working_dir.parent)
     if (p / "lfm/data_processing").is_dir() and (p / "TMS").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Run from the repository root or top-level notebooks/ directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lfm.data_processing.raster_coverage import (
    inspect_raster_footprint,
    rank_raster_pairs,
)

print(f"Repository: {repo_root}")

## 2. User configuration

There is **no fixed AOI**. Each WAC file supplies its own full footprint and
area denominator. `RASTER_GLOB` searches immediately inside each directory;
change to `**/*.tif` only if you want nested directories too. VIS and UV are
identified from `.vis.` and `.uv.` in their filenames; any other WAC TIFFs
are still scanned and ranked.

Use a session/allocation with at least 10 CPUs for `MAX_WORKERS = 10`.
The footprint scan and ranking each use this setting. Ranking builds one NAC
spatial index per worker; lowering this setting reduces replicated index memory.
`MAX_WORKERS = 1` runs ranking serially for comparisons.
`EDGE_SEGMENTS` controls boundary approximation. Ordinary scene footprints,
including antimeridian crossings, are supported; pole-enclosing/global rasters
are reported as errors. Output storage and dictionary memory grow as M × N.

In [ ]:
NAC_DATA_DIRS = [
    Path("/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_IMP"),
    Path("/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_Pho_Sites"),
]
WAC_DATA_DIR = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_WAC_Pho_Sites")
RASTER_GLOB = "*.tif"
MAX_WORKERS = 10
EDGE_SEGMENTS = 64
OUTPUT_BASE_DIR = repo_root / "notebooks/outputs/nac_aoi_overlap"

## 3. Discover all WAC and NAC files

Full paths are dictionary keys so matching basenames in the two NAC
collections remain distinct. Missing directories stop the scan. Every
discovered WAC and NAC remains represented in the final matrix, including
files that fail inspection. The cell prints the required M × N entry count.

In [ ]:
LUNAR_WKT_PATH = repo_root / "TMS/IAU_30100_2015.wkt"
lunar_wkt = LUNAR_WKT_PATH.read_text(encoding="utf-8")
if not isinstance(MAX_WORKERS, int) or MAX_WORKERS < 1:
    raise ValueError("MAX_WORKERS must be a positive integer.")
if not isinstance(EDGE_SEGMENTS, int) or EDGE_SEGMENTS < 4:
    raise ValueError("EDGE_SEGMENTS must be an integer >= 4.")

source_directories = [(directory, "nac") for directory in NAC_DATA_DIRS]
source_directories.append((WAC_DATA_DIR, "wac"))
missing_dirs = [str(path) for path, _ in source_directories if not path.is_dir()]
if missing_dirs:
    raise FileNotFoundError("Source directories unavailable: " + ", ".join(missing_dirs))

file_groups = {}
for directory, modality in source_directories:
    found = sorted(path.absolute() for path in directory.glob(RASTER_GLOB) if path.is_file())
    print(f"{directory}: {len(found):,} matching TIFFs")
    if not found:
        print("  No files matched in this collection.")
    for path in found:
        group = modality
        if modality == "wac":
            name = path.name.lower()
            group = "wac_vis" if ".vis." in name else "wac_uv" if ".uv." in name else "wac_other"
        filename = str(path)
        if filename in file_groups and file_groups[filename] != group:
            raise ValueError(f"Source configured under conflicting modalities: {filename}")
        file_groups[filename] = group
source_files = sorted(file_groups)
wac_files = [path for path in source_files if file_groups[path].startswith("wac_")]
nac_files = [path for path in source_files if file_groups[path] == "nac"]
if not wac_files or not nac_files:
    raise FileNotFoundError("Need at least one WAC and one NAC TIFF. Check source directories and RASTER_GLOB.")

GROUP_LABELS = {"nac": "NAC", "wac_vis": "WAC VIS", "wac_uv": "WAC UV", "wac_other": "WAC other"}
for group, label in GROUP_LABELS.items():
    print(f"{label}: {sum(value == group for value in file_groups.values()):,} files")
print(f"M = {len(wac_files):,} WAC; N = {len(nac_files):,} NAC")
print(f"Nested dictionary will contain {len(wac_files) * len(nac_files):,} total entries.")

## 4. Read full footprints in parallel

Each worker opens and closes its own raster handle, without reading image
bands or masks. The parent collects the full geographic footprints in
`raster_footprints` and error messages in `scan_errors`. Footprints are read
once and reused for all pair comparisons. A failed file has a `None` footprint;
its filename remains in the matrix. Rerunning this cell resets the scan.

In [ ]:
raster_footprints = {filename: None for filename in source_files}
scan_errors = {}
scan_started_utc = datetime.now(timezone.utc).isoformat()
worker = partial(
    inspect_raster_footprint,
    lunar_wkt=lunar_wkt,
    edge_segments=EDGE_SEGMENTS,
)

with ProcessPoolExecutor(
    max_workers=MAX_WORKERS,
    mp_context=multiprocessing.get_context("spawn"),
) as pool:
    results = pool.map(worker, source_files, chunksize=8)
    for completed, result in enumerate(results, start=1):
        filename = result["filename"]
        if result["error"] is not None:
            scan_errors[filename] = result["error"]
        else:
            raster_footprints[filename] = result["footprint_wkb_hex"]
        if completed == 1 or completed % 100 == 0 or completed == len(source_files):
            print(f"Checked {completed:,}/{len(source_files):,} files; errors: {len(scan_errors):,}", flush=True)

scan_finished_utc = datetime.now(timezone.utc).isoformat()
print(f"Inspected successfully: {len(source_files) - len(scan_errors):,}; failed: {len(scan_errors):,}")
if scan_errors:
    print("First five errors (all errors are saved below):")
    for filename, error in list(scan_errors.items())[:5]:
        print(f"  {filename}: {error}")

## 5. Build the M × N dictionary, save results, and print the top five overall

`overlap_by_wac[wac_filename][nac_filename]` gives the precise percentage of
**that full WAC footprint** covered by the NAC footprint. Every WAC entry has
exactly N NAC entries, including `0.0` for disjoint or boundary-touching files.
Unreadable files produce `None`, not zero; in JSON these become `null`.

Ranking distributes WAC files across `MAX_WORKERS` processes. Each builds the
NAC spatial index once and returns only positive overlaps plus its top pairs.
The parent fills the complete M × N dictionary, including all zeros and `None`
values. Only a bounded number of jobs are pending at once. A regular `tqdm`
progress bar advances once per completed WAC, including failed-input rows.
Its initial wait includes worker startup and NAC index construction. The top five come from
all positive pairs, including both VIS and UV; ties use WAC then NAC filename.
Shared area in km² is also printed so you can judge the size of a poster AOI.

Outputs in a new run directory:

- `overlap_by_wac.json`: the complete nested dictionary with precise percentages.
- `overlap_int_by_wac.json`: the same structure with rounded integer percentages;
  a small positive overlap may round to zero, so ranking uses the precise values.
- `scan_report.json`: sources, file groups, errors, method, counts, and top pairs.

JSON is written one row at a time, with progress and a separate elapsed time
for each matrix file. The required nested dictionary itself stays in memory.
The report records ranking wall time, parent dictionary-construction time,
and cumulative worker times for index building and intersections. Worker
times run concurrently and must not be added to the wall time. Index timing
counts each worker that returned a WAC result once.

If you updated the helper in an active kernel, reload it before rerunning this
cell to reuse the existing `raster_footprints` without rescanning:

```python
import importlib
import lfm.data_processing.raster_coverage as coverage
importlib.reload(coverage)
rank_raster_pairs = coverage.rank_raster_pairs
```

In [ ]:
print("Comparing full WAC and NAC footprints...", flush=True)
with tqdm(total=len(wac_files), desc="Ranking WAC–NAC overlaps", unit="WAC") as ranking_progress:
    ranking = rank_raster_pairs(
        {path: raster_footprints[path] for path in wac_files},
        {path: raster_footprints[path] for path in nac_files},
        lunar_wkt=lunar_wkt,
        limit=5,
        max_workers=MAX_WORKERS,
        progress_callback=lambda completed, total: ranking_progress.update(completed - ranking_progress.n),
    )
comparison_finished_utc = datetime.now(timezone.utc).isoformat()
overlap_by_wac = ranking["overlap_percent_by_wac"]
top_five = ranking["top_pairs"]
assert len(overlap_by_wac) == len(wac_files)
assert all(len(row) == len(nac_files) for row in overlap_by_wac.values())
print(f"Matrix: {len(wac_files):,} × {len(nac_files):,}; positive pairs: {ranking['positive_pair_count']:,}")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = OUTPUT_BASE_DIR / run_id
output_dir.mkdir(parents=True, exist_ok=False)
def write_overlap_matrix(filename, *, rounded=False):
    started = perf_counter()
    last_update = started
    print(f"Writing {filename}...", flush=True)
    with (output_dir / filename).open("w", encoding="utf-8") as stream:
        stream.write("{\n")
        for completed, (wac_filename, row) in enumerate(overlap_by_wac.items(), start=1):
            if completed > 1:
                stream.write(",\n")
            stream.write(json.dumps(wac_filename) + ": ")
            output_row = ({nac: None if percent is None else int(round(percent))
                           for nac, percent in row.items()} if rounded else row)
            json.dump(output_row, stream, allow_nan=False)
            now = perf_counter()
            if completed == 1 or completed == len(overlap_by_wac) or now - last_update >= 5:
                print(f"  {completed:,}/{len(overlap_by_wac):,} WAC rows; {now - started:.1f}s", flush=True)
                last_update = now
        stream.write("\n}\n")
    return perf_counter() - started

output_timings = {
    "precise_json_seconds": write_overlap_matrix("overlap_by_wac.json"),
    "integer_json_seconds": write_overlap_matrix("overlap_int_by_wac.json", rounded=True),
}
report = {
    "search_scope": "full WAC footprints; no fixed AOI",
    "lunar_geographic_wkt": lunar_wkt,
    "coverage_definition": "100 * area(WAC footprint intersect NAC footprint) / area(WAC footprint)",
    "area_method": "lunar spherical geodesic areas of densified geographic boundaries",
    "includes_nodata_margins": True,
    "edge_segments": EDGE_SEGMENTS,
    "source_directories": [{"path": str(path), "modality": modality} for path, modality in source_directories],
    "raster_glob": RASTER_GLOB,
    "worker_count": MAX_WORKERS,
    "scan_started_utc": scan_started_utc,
    "scan_finished_utc": scan_finished_utc,
    "comparison_finished_utc": comparison_finished_utc,
    "ranking_workers_used": ranking["workers_used"],
    "ranking_timings": ranking["timings"],
    "output_timings": output_timings,
    "wac_count": len(wac_files),
    "nac_count": len(nac_files),
    "matrix_entry_count": len(wac_files) * len(nac_files),
    "positive_pair_count": ranking["positive_pair_count"],
    "file_groups": file_groups,
    "overlap_matrix_file": "overlap_by_wac.json",
    "top_five": top_five,
    "errors": scan_errors,
}
(output_dir / "scan_report.json").write_text(
    json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8"
)
print(f"Saved scan to: {output_dir}")
print(f"Ranking elapsed: {ranking['timings']['ranking_wall_seconds']:.1f}s; "
      f"parent dictionary construction: {ranking['timings']['matrix_build_seconds']:.1f}s")
print(f"Cumulative worker time: index {ranking['timings']['worker_index_seconds']:.1f}s; "
      f"intersections {ranking['timings']['worker_intersection_seconds']:.1f}s")
print(f"JSON output: precise {output_timings['precise_json_seconds']:.1f}s; "
      f"rounded {output_timings['integer_json_seconds']:.1f}s")
print("\nTop five pairs overall, by percentage of the full WAC footprint covered:")
for rank, pair in enumerate(top_five, start=1):
    wac_type = GROUP_LABELS[file_groups[pair["wac_filename"]]]
    print(f"{rank}. {pair['wac_coverage_percent']:.6f}% of {wac_type}; {pair['shared_area_km2']:.3f} km² shared")
    print(f"   WAC: {pair['wac_filename']}")
    print(f"   NAC: {pair['nac_filename']}")
if not top_five:
    print("No positive overlaps found. Check scan_errors before concluding there is no coverage.")

## 5a. Find a shared AOI for the selected WAC–NAC pair

This cell uses the cached footprints from Step 4. If a selected file is not in
that cache, it reads metadata for **only that file**; no collection scan or
ranking is needed. You can also run it after the imports cell alone.

It prints two IAU:30100 tuples in **north, west, south, east** order:

- `OVERLAP_BOUNDS_NWSE`: the enclosing rectangle of the complete intersection.
  Its corners may extend beyond the shared footprint.
- `AOI_NWSE`: a conservative rectangle contained in the shared footprint,
  centered on an interior point. It is a starting AOI, not the largest possible
  rectangle. The cell shrinks it slightly to leave a margin from the boundary.

Longitude wrapping is handled; west > east denotes an antimeridian crossing.
The calculation uses raster footprints, including NoData margins. Inspect
valid imagery and choose a suitably small AOI before creating NAC-resolution
chips; native-grid rounding can also expand the requested rectangle.
This pair uses **WAC UV**. Check the same-product VIS companion before using
this AOI for seven-band WAC chip creation.

In [ ]:
from shapely.affinity import translate
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.wkb import loads as load_wkb

PAIR_WAC = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_WAC_Pho_Sites/M155084577CE.prj.uv.mos.tif")
PAIR_NAC = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_IMP/M1456065428RE.ech.cog.tif")

# Clear previous results so a failed rerun cannot leave an old AOI behind.
AOI_NWSE = None
OVERLAP_BOUNDS_NWSE = None
pair_overlap = None
pair_aoi_rectangle = None
pair_lunar_wkt = (repo_root / "TMS/IAU_30100_2015.wkt").read_text(encoding="utf-8")

def selected_pair_footprint(path):
    cached = globals().get("raster_footprints", {}).get(str(path))
    if cached is None:
        result = inspect_raster_footprint(
            path, lunar_wkt=pair_lunar_wkt,
            edge_segments=globals().get("EDGE_SEGMENTS", 64),
        )
        if result["error"] is not None:
            raise ValueError(f"Cannot inspect {path}: {result['error']}")
        cached = result["footprint_wkb_hex"]
    return load_wkb(cached, hex=True)

pair_wac_footprint = selected_pair_footprint(PAIR_WAC)
pair_nac_footprint = selected_pair_footprint(PAIR_NAC)
# Match the longitude copies used by ranking, including antimeridian scenes.
pair_intersections = [
    pair_wac_footprint.intersection(translate(pair_nac_footprint, xoff=shift))
    for shift in (-360, 0, 360)
]
pair_intersections = [part for part in pair_intersections if not part.is_empty and part.area > 0]
if not pair_intersections:
    raise ValueError("The selected WAC and NAC footprints have no positive-area overlap.")
pair_overlap = unary_union(pair_intersections)
west, south, east, north = pair_overlap.bounds

def bounds_to_nwse(bounds):
    west, south, east, north = bounds
    return (north, (west + 180) % 360 - 180, south, (east + 180) % 360 - 180)

# Use an interior point rather than a centroid, which may lie outside a
# concave or multipart intersection. Search for a contained rectangle with
# fixed aspect ratio at this center; this is not a global maximum-area search.
center = pair_overlap.representative_point()
half_width = min(center.x - west, east - center.x)
half_height = min(center.y - south, north - center.y)
if half_width <= 0 or half_height <= 0:
    raise ValueError("Cannot construct a positive-area rectangular AOI from this intersection.")
low, high = 0.0, 1.0
for _ in range(48):
    scale = (low + high) / 2
    candidate = box(center.x - scale * half_width, center.y - scale * half_height,
                    center.x + scale * half_width, center.y + scale * half_height)
    if pair_overlap.covers(candidate):
        low = scale
    else:
        high = scale
scale = low * 0.99
pair_aoi_rectangle = box(center.x - scale * half_width, center.y - scale * half_height,
                         center.x + scale * half_width, center.y + scale * half_height)
if pair_aoi_rectangle.area <= 0 or not pair_overlap.covers(pair_aoi_rectangle):
    raise ValueError("Could not find a contained rectangular AOI; inspect pair_overlap directly.")

OVERLAP_BOUNDS_NWSE = bounds_to_nwse(pair_overlap.bounds)
AOI_NWSE = bounds_to_nwse(pair_aoi_rectangle.bounds)
print(f"WAC: {PAIR_WAC}\nNAC: {PAIR_NAC}")
print("\nIAU:30100 degrees, ordered north, west, south, east:")
print("OVERLAP_BOUNDS_NWSE = (" + ", ".join(f"{v:.10f}" for v in OVERLAP_BOUNDS_NWSE) + ")")
print(f"Full bounding rectangle contained in overlap: {pair_overlap.covers(box(*pair_overlap.bounds))}")
print("\nContained starting AOI for chip creation:")
print("AOI_NWSE = (" + ", ".join(f"{v:.10f}" for v in AOI_NWSE) + ")")
print(f"Crosses antimeridian: {AOI_NWSE[1] > AOI_NWSE[3]}")
print("Inspect valid pixels and reduce the AOI as needed before creating NAC-resolution chips.")

## 6. Use a pair for the poster

1. Choose a pair from `top_five` and inspect both images, including NoData
   margins. Identify a smaller common geographic AOI inside valid imagery.
2. For seven-band WAC chip creation, use a VIS image and its **same-product-ID**
   UV companion. VIS and UV files are ranked independently here; a high-ranked
   UV file from a different product is not the VIS image's companion. The two
   example filenames in the request have different IDs. If a UV–NAC pair wins,
   locate that UV image's matching VIS file and inspect its overlap too.
   This pair search does not prove simultaneous VIS + UV + NAC valid coverage.
3. Label the chosen NAC image using [crater_labeling.ipynb](crater_labeling.ipynb),
   setting `DEFAULT_RASTER` to its full path. Save the finished annotations.
4. In [chip_example.ipynb](chip_example.ipynb), run WAC and NAC separately with
   their respective `SOURCE_RASTER`, finished `LABEL_PATH`, and the **same small
   AOI_NWSE**. The two runs keep source-native resolutions; the same geographic
   AOI does not imply equal pixel dimensions. Use those runs' inspection PNGs.

Avoid using an entire WAC scene as a single NAC-resolution chip: choose a
poster-sized region inside the shared valid footprint. An unlabeled region
outside NAC coverage is not confirmed crater-free ground.